In [1]:
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/sasha/miniconda3/envs/lima/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_PATH = Path("../data/qwen_train_data.jsonl")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 201.44it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [3]:
examples = []

with DATA_PATH.open("r", encoding="utf-8") as f:
    for i in range(2):
        examples.append(json.loads(f.readline()))

for i, example in enumerate(examples):
    print(f"\nExample {i + 1}")
    print(json.dumps(example, indent=2, ensure_ascii=False))


Example 1
{
  "text": "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nCan brain cells move? By movement I mean long distance migration (preferably within the brain only).<|im_end|>\n<|im_start|>assistant\nThe question is relatively broad and one should take into account that the brain not only consists of neurons, but also glial cells (supportive cells) and pre-mitotic neuronal stem cells. Furthermore, as critical fellow-scientists have indicated, developmental stage is very important, as the developing embryonic brain is very different from the adult brain.\nHowever, after sifting through various publications, the answer to the question is actually remarkably simple: Yes, brain cells migrate.\nIn  the adult brain glial cells migrate in the brain (Klämbt, 2009). Glial cells are involved in a myriad of functions, but a notable example of migrating glial cells are the oligodendrocytes that migrate relative long dista

In [4]:
texts = [example["text"] for example in examples]

batch = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
)

print(f"Batch fields: {list(batch.keys())}")
print(f"Input IDs shape: {tuple(batch['input_ids'].shape)}")
print(f"Attention mask shape: {tuple(batch['attention_mask'].shape)}")

Batch fields: ['input_ids', 'attention_mask']
Input IDs shape: (2, 822)
Attention mask shape: (2, 822)


In [5]:
labels = batch["input_ids"].clone()
labels[batch["attention_mask"] == 0] = -100
ignored_positions = (labels == -100).sum().item()

print(f"Labels shape      : {tuple(labels.shape)}")
print(f"Ignored positions : {ignored_positions}")

Labels shape      : (2, 822)
Ignored positions : 411


In [9]:
with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=labels,
    )

print(f"Forward Pass Output Fields: {outputs.keys()}")

Forward Pass Output Fields: odict_keys(['loss', 'logits', 'past_key_values'])


In [10]:
print(f"Loss: {outputs.loss.item():.6f}")
print(f"Logits shape: {tuple(outputs.logits.shape)}")

Loss: 2.995932
Logits shape: (2, 822, 151936)


In [11]:
row = 0
pos = 20

pred_id = outputs.logits[row, pos].argmax().item()

current = tokenizer.decode([batch["input_ids"][row, pos].item()])
pred = tokenizer.decode([pred_id])

true_id = labels[row, pos + 1].item()
true = "<ignored>" if true_id == -100 else tokenizer.decode([true_id])

print("Current token:", repr(current))
print("Predicted next token:", repr(pred))
print("Actual next token:", repr(true))

Current token: '\n'
Predicted next token: 'Q'
Actual next token: '<|im_start|>'
